# MVRV-Based Scale Out Exit Strategies

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from numba import njit
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path("../data/daily")
price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
mvrv = pd.read_parquet(DATA_DIR / "mvrv.parquet").rename(columns={"value": "mvrv"}).set_index("time")
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
realized_loss = pd.read_parquet(DATA_DIR / "realized_loss.parquet").rename(columns={"value": "realized_loss"}).set_index("time")

df = price.join(mvrv, how='inner').join(sopr, how='inner').join(sopr_sth, how='inner').join(realized_loss, how='inner')
df = df.sort_index()
df['rl_zscore'] = (df['realized_loss'] - df['realized_loss'].rolling(30).mean()) / df['realized_loss'].rolling(30).std()
df = df[df.index >= '2018-12-15'].dropna()
print(f"Data: {len(df)} rows ({df.index.min().date()} to {df.index.max().date()})")

In [ ]:
entry_condition = (df['sopr'] < 1) & (df['sopr_sth'] < 1) & (df['rl_zscore'] > 0.5)
entries = entry_condition & ~entry_condition.shift(1).fillna(False)
print(f"Entry signals: {entries.sum()}")

In [ ]:
@njit
def exit_simple_trail(price_arr, mvrv_arr, entry_idx, trail_pct=0.30, stop_loss=0.20):
    entry_price = price_arr[entry_idx]
    peak = entry_price
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        if price > peak: peak = price
        if price <= peak * (1 - trail_pct): return j, price
        if (price - entry_price) / entry_price <= -stop_loss: return j, price
    return len(price_arr) - 1, price_arr[-1]

@njit
def exit_mvrv_scale_3(price_arr, mvrv_arr, entry_idx, mvrv1=2.0, mvrv2=2.5, mvrv3=3.0, trail_pct=0.30, stop_loss=0.20):
    entry_price = price_arr[entry_idx]
    peak = entry_price
    t1_closed, t2_closed = False, False
    t1_exit, t2_exit = 0.0, 0.0
    for j in range(entry_idx + 1, len(price_arr)):
        price, mvrv = price_arr[j], mvrv_arr[j]
        if price > peak: peak = price
        if not t1_closed and mvrv > mvrv1: t1_closed, t1_exit = True, price
        if not t2_closed and mvrv > mvrv2: t2_closed, t2_exit = True, price
        if t1_closed and t2_closed and mvrv > mvrv3:
            return j, (t1_exit + t2_exit + price) / 3
        if price <= peak * (1 - trail_pct):
            exits = [t1_exit if t1_closed else price, t2_exit if t2_closed else price, price]
            return j, sum(exits) / 3
        if (price - entry_price) / entry_price <= -stop_loss: return j, price
    exits = [t1_exit if t1_closed else price_arr[-1], t2_exit if t2_closed else price_arr[-1], price_arr[-1]]
    return len(price_arr) - 1, sum(exits) / 3

@njit
def exit_mvrv_scale_2(price_arr, mvrv_arr, entry_idx, mvrv1=2.5, mvrv2=3.0, trail_pct=0.30, stop_loss=0.20):
    entry_price = price_arr[entry_idx]
    peak = entry_price
    t1_closed, t1_exit = False, 0.0
    for j in range(entry_idx + 1, len(price_arr)):
        price, mvrv = price_arr[j], mvrv_arr[j]
        if price > peak: peak = price
        if not t1_closed and mvrv > mvrv1: t1_closed, t1_exit = True, price
        if t1_closed and mvrv > mvrv2: return j, (t1_exit + price) / 2
        if price <= peak * (1 - trail_pct): return j, (t1_exit + price) / 2 if t1_closed else price
        if (price - entry_price) / entry_price <= -stop_loss: return j, price
    return len(price_arr) - 1, (t1_exit + price_arr[-1]) / 2 if t1_closed else price_arr[-1]

@njit
def exit_mvrv_trail_trigger(price_arr, mvrv_arr, entry_idx, mvrv_trigger=2.0, trail_pct=0.25, stop_loss=0.20):
    entry_price = price_arr[entry_idx]
    peak = entry_price
    trail_active = False
    for j in range(entry_idx + 1, len(price_arr)):
        price, mvrv = price_arr[j], mvrv_arr[j]
        if price > peak: peak = price
        if not trail_active and mvrv > mvrv_trigger: trail_active = True
        if trail_active and price <= peak * (1 - trail_pct): return j, price
        if not trail_active and (price - entry_price) / entry_price <= -stop_loss: return j, price
    return len(price_arr) - 1, price_arr[-1]

@njit
def exit_mvrv_tighten(price_arr, mvrv_arr, entry_idx, mvrv_tight=2.5, trail1=0.30, trail2=0.15, stop_loss=0.20):
    entry_price = price_arr[entry_idx]
    peak = entry_price
    trail_pct = trail1
    for j in range(entry_idx + 1, len(price_arr)):
        price, mvrv = price_arr[j], mvrv_arr[j]
        if price > peak: peak = price
        if mvrv > mvrv_tight: trail_pct = trail2
        if price <= peak * (1 - trail_pct): return j, price
        if (price - entry_price) / entry_price <= -stop_loss: return j, price
    return len(price_arr) - 1, price_arr[-1]

In [ ]:
def run_backtest(df, entries, exit_func, initial_capital=100000, **kwargs):
    price_arr, mvrv_arr, dates = df['price'].values, df['mvrv'].values, df.index
    entry_indices = np.where(entries.values)[0]
    trades, i = [], 0
    while i < len(entry_indices):
        entry_idx = entry_indices[i]
        exit_idx, exit_price = exit_func(price_arr, mvrv_arr, entry_idx, **kwargs)
        entry_price = price_arr[entry_idx]
        net_return = (exit_price / entry_price) - 1 - 0.002
        trades.append({'entry_date': dates[entry_idx], 'exit_date': dates[exit_idx],
                       'entry_price': entry_price, 'exit_price': exit_price,
                       'entry_mvrv': mvrv_arr[entry_idx], 'exit_mvrv': mvrv_arr[exit_idx],
                       'net_return': net_return})
        while i < len(entry_indices) and entry_indices[i] <= exit_idx: i += 1
    trades_df = pd.DataFrame(trades)
    if len(trades_df) > 0:
        equity = [initial_capital]
        for _, t in trades_df.iterrows(): equity.append(equity[-1] * (1 + t['net_return']))
        trades_df['equity'] = equity[1:]
    return trades_df

def calc_metrics(trades, initial_capital):
    if len(trades) == 0: return {'total_return': 0, 'sharpe': 0, 'max_dd': 0, 'win_rate': 0}
    final_equity = trades['equity'].iloc[-1]
    total_return = (final_equity / initial_capital) - 1
    years = (trades['exit_date'].iloc[-1] - trades['entry_date'].iloc[0]).days / 365.25
    win_rate = (trades['net_return'] > 0).mean()
    returns = trades['net_return'].values
    sharpe = (returns.mean() / returns.std()) * np.sqrt(len(trades)/years) if returns.std() > 0 else 0
    equity = [initial_capital] + list(trades['equity'])
    peak, max_dd = equity[0], 0
    for eq in equity:
        if eq > peak: peak = eq
        dd = (eq - peak) / peak
        if dd < max_dd: max_dd = dd
    return {'total_return': total_return, 'sharpe': sharpe, 'max_dd': max_dd, 'win_rate': win_rate, 'final_equity': final_equity}

In [ ]:
strategies = [
    ('1. Simple 30% Trail', exit_simple_trail, {'trail_pct': 0.30}),
    ('2. MVRV Scale 3x (2.0/2.5/3.0)', exit_mvrv_scale_3, {'mvrv1': 2.0, 'mvrv2': 2.5, 'mvrv3': 3.0}),
    ('3. MVRV Scale 3x (2.5/3.0/3.5)', exit_mvrv_scale_3, {'mvrv1': 2.5, 'mvrv2': 3.0, 'mvrv3': 3.5}),
    ('4. MVRV Scale 3x (1.8/2.2/2.6)', exit_mvrv_scale_3, {'mvrv1': 1.8, 'mvrv2': 2.2, 'mvrv3': 2.6}),
    ('5. MVRV Scale 2x (2.0/2.5)', exit_mvrv_scale_2, {'mvrv1': 2.0, 'mvrv2': 2.5}),
    ('6. MVRV Scale 2x (2.5/3.0)', exit_mvrv_scale_2, {'mvrv1': 2.5, 'mvrv2': 3.0}),
    ('7. MVRV Scale 2x (2.0/3.0)', exit_mvrv_scale_2, {'mvrv1': 2.0, 'mvrv2': 3.0}),
    ('8. MVRV Trail (trigger 2.0)', exit_mvrv_trail_trigger, {'mvrv_trigger': 2.0, 'trail_pct': 0.25}),
    ('9. MVRV Trail (trigger 2.5)', exit_mvrv_trail_trigger, {'mvrv_trigger': 2.5, 'trail_pct': 0.20}),
    ('10. MVRV Tighten (2.5: 30->15%)', exit_mvrv_tighten, {'mvrv_tight': 2.5, 'trail1': 0.30, 'trail2': 0.15}),
    ('11. MVRV Tighten (2.0: 30->20%)', exit_mvrv_tighten, {'mvrv_tight': 2.0, 'trail1': 0.30, 'trail2': 0.20}),
]

results = []
print('MVRV EXIT STRATEGY COMPARISON')
print('='*120)
print(f"{'Strategy':<35} {'Return':>12} {'Sharpe':>8} {'Win%':>8} {'MaxDD':>10} {'Final $':>14}")
print('-'*120)

for name, func, kwargs in strategies:
    trades = run_backtest(df, entries, func, 100000, **kwargs)
    m = calc_metrics(trades, 100000)
    print(f"{name:<35} {m['total_return']*100:>+11.0f}% {m['sharpe']:>8.2f} {m['win_rate']*100:>7.0f}% {m['max_dd']*100:>9.0f}% ${m['final_equity']:>13,.0f}")
    results.append({'name': name, 'trades': trades, 'metrics': m})

In [ ]:
for r in results:
    m = r['metrics']
    r['score'] = (m['total_return'] * m['sharpe']) / abs(m['max_dd']) if m['max_dd'] != 0 else 0

by_score = sorted(results, key=lambda x: x['score'], reverse=True)
print('\nRANKING (Return x Sharpe / MaxDD)')
print('='*60)
for i, r in enumerate(by_score[:6]):
    m = r['metrics']
    print(f"{i+1}. {r['name']}: Return {m['total_return']*100:+,.0f}%, Sharpe {m['sharpe']:.2f}, MaxDD {m['max_dd']*100:.0f}%")

In [ ]:
best_mvrv = [r for r in by_score if 'MVRV' in r['name']][0]
print(f"\nBest MVRV Strategy: {best_mvrv['name']}")
print('='*100)
for _, t in best_mvrv['trades'].iterrows():
    print(f"{t['entry_date'].date()} (MVRV {t['entry_mvrv']:.2f}) -> {t['exit_date'].date()} (MVRV {t['exit_mvrv']:.2f}): ${t['entry_price']:,.0f} -> ${t['exit_price']:,.0f} = {t['net_return']*100:+.0f}%")